In [1]:
import pandas as pd

In [17]:
house_prices = pd.read_csv(
    "house_price_index_europe.csv",
    encoding="cp1252"
)

In [ ]:
print(house_prices.head())


                                        Year   2015 Unnamed: 2    2016  \
0                                  Countries    NaN        NaN     NaN   
1  European Union - 27 countries (from 2020)  100.0        NaN  104.31   
2       Euro area ? 21 countries (from 2026)  100.0        NaN  104.04   
3       Euro area ? 20 countries (2023-2025)  100.0        NaN  104.01   
4                                    Belgium  100.0        NaN  102.33   

  Unnamed: 4    2017 Unnamed: 6    2018 Unnamed: 8    2019  ...    2020  \
0        NaN     NaN        NaN     NaN        NaN     NaN  ...     NaN   
1        NaN  109.31        NaN  114.81        NaN  120.40  ...  127.14   
2        NaN  108.66        NaN  113.92        NaN  118.96  ...  125.23   
3        NaN  108.59        NaN  113.83        NaN  118.84  ...  125.11   
4        NaN  105.95        NaN  109.15        NaN  113.23  ...  118.09   

   Unnamed: 12    2021  Unnamed: 14    2022  Unnamed: 16    2023  Unnamed: 18  \
0          NaN     NaN 

In [20]:
house_prices = house_prices.loc[:, ~house_prices.columns.str.contains('Unnamed')]

In [21]:
print(house_prices.columns)

Index(['Year', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024'],
      dtype='str')


In [22]:
house_prices.rename(columns={'Year': 'country'}, inplace=True)

In [23]:
house_prices = house_prices[
    ~house_prices['country'].str.contains('European|Euro', na=False)
]

In [24]:
house_prices_long = house_prices.melt(
    id_vars='country',
    var_name='year',
    value_name='house_price_index'
)

In [25]:
house_prices_long['house_price_index'] = (
    house_prices_long['house_price_index']
    .astype(str)
    .str.extract(r'([\d\.]+)')
)

In [26]:
house_prices_long['year'] = pd.to_numeric(
    house_prices_long['year'], errors='coerce'
)

house_prices_long['house_price_index'] = pd.to_numeric(
    house_prices_long['house_price_index'], errors='coerce'
)

In [27]:
house_prices_long = house_prices_long.dropna()

In [28]:
house_prices_long.to_csv(
    "house_prices_clean.csv",
    index=False
)

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import mysql.connector

# Connect to MySQL
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="housing_project"
)

ProgrammingError: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)

In [ ]:
query1 = """
SELECT 
    country,
    MIN(house_price_index) AS starting_price,
    MAX(house_price_index) AS latest_price
FROM house_prices_clean
GROUP BY country
ORDER BY latest_price DESC
"""

df1 = pd.read_sql(query1, connection)

# Calculate growth
df1["growth"] = df1["latest_price"] - df1["starting_price"]

# Plot
plt.figure()
plt.bar(df1["country"], df1["growth"])

plt.title("Housing Price Growth by Country")
plt.xlabel("Country")
plt.ylabel("Price Index Growth")

plt.xticks(rotation=90)

plt.show()